## 1. Imports

In [1]:
import time
from collections import deque
from itertools   import product as cart_product


## 2. Circuit Representation

A circuit is modelled as a directed graph:

```
GateNode  ──fanout──►  GateNode  ──fanout──►  output
          ◄──fanin──
```

Three classes are used:
- `GateNode`   — every gate, wire, or primary I/O point
- `StuckFault` — one SA0 or SA1 fault at a specific node
- `LogicNet`   — the complete circuit container


In [2]:
class GateNode:
    """
    One node in the circuit graph (could be a PI, gate output, or PO).
    Carries a 5-valued logic assignment during ATPG.
    """
    def __init__(self, label, kind, category='GATE'):
        self.label    = label      # e.g. 'G3', 'a', 'out'
        self.kind     = kind       # AND | OR | NAND | NOR | NOT | BUF | XOR | XNOR | PI
        self.category = category  # PI | PO | GATE | CONST
        self.drivers  = []         # nodes feeding INTO this node
        self.loads    = []         # nodes this node feeds INTO
        self.depth    = 0          # topological depth from inputs
        self.sig      = 'X'       # current 5-valued signal

    def __repr__(self):
        return f"<{self.label}:{self.kind} sig={self.sig}>"


class StuckFault:
    """Represents a single stuck-at fault on one node."""
    def __init__(self, target_node, stuck_val):
        self.target    = target_node   # GateNode object
        self.stuck_val = stuck_val     # 0 or 1

    def __repr__(self):
        return f"StuckFault({self.target.label} SA{self.stuck_val})"


class LogicNet:
    """Holds the complete circuit — all nodes, inputs, and outputs."""
    def __init__(self):
        self.node_map = {}   # label -> GateNode
        self.inputs   = []   # primary inputs (ordered)
        self.outputs  = []   # primary outputs (ordered)

    def register(self, node):
        self.node_map[node.label] = node

print("✅ GateNode, StuckFault, LogicNet defined.")


✅ GateNode, StuckFault, LogicNet defined.


## 3. Netlist Input Format

Circuits are described as a plain Python dictionary — no file I/O needed.

```python
my_circuit = {
    "inputs":  ["A", "B", "C"],
    "outputs": ["Z"],
    "gates": [
        # (wire_name, gate_type, [input_wire_names])
        ("w1", "AND",  ["A", "B"]),
        ("Z",  "OR",   ["w1", "C"]),
    ]
}
```

Valid gate types: `AND`, `OR`, `NAND`, `NOR`, `NOT`, `BUF`, `XOR`, `XNOR`


In [3]:
def netlist_to_graph(spec: dict) -> LogicNet:
    """
    Convert a netlist dictionary into a LogicNet graph.
    Validates that all referenced wires exist before wiring up connections.
    """
    net = LogicNet()

    # 1. Register all primary inputs
    for lbl in spec['inputs']:
        nd = GateNode(lbl, 'PI', category='PI')
        net.register(nd)
        net.inputs.append(nd)

    # 2. Register gate output nodes (first pass — no wiring yet)
    for (out_lbl, gate_kind, _) in spec['gates']:
        if out_lbl not in net.node_map:
            nd = GateNode(out_lbl, gate_kind, category='GATE')
            net.register(nd)
        else:
            net.node_map[out_lbl].kind = gate_kind

    # 3. Wire up driver → load relationships
    for (out_lbl, _, in_lbls) in spec['gates']:
        out_nd = net.node_map[out_lbl]
        for in_lbl in in_lbls:
            if in_lbl not in net.node_map:
                raise KeyError(f"Wire '{in_lbl}' (input to '{out_lbl}') was never declared.")
            in_nd = net.node_map[in_lbl]
            out_nd.drivers.append(in_nd)
            in_nd.loads.append(out_nd)

    # 4. Tag primary outputs
    for lbl in spec['outputs']:
        if lbl not in net.node_map:
            raise KeyError(f"Output wire '{lbl}' not found in gate definitions.")
        net.node_map[lbl].category = 'PO'
        net.outputs.append(net.node_map[lbl])

    return net


def assign_depths(net: LogicNet):
    """BFS from primary inputs to assign topological depth to every node."""
    for nd in net.node_map.values():
        nd.depth = 0
    frontier = deque(net.inputs)
    seen     = {nd.label for nd in net.inputs}
    while frontier:
        nd = frontier.popleft()
        for ld in nd.loads:
            ld.depth = max(ld.depth, nd.depth + 1)
            if ld.label not in seen:
                seen.add(ld.label)
                frontier.append(ld)


def enumerate_faults(net: LogicNet) -> list:
    """Generate the full SA0/SA1 fault list for every non-constant node."""
    flist = []
    for nd in net.node_map.values():
        if nd.category == 'CONST':
            continue
        flist.append(StuckFault(nd, 0))
        flist.append(StuckFault(nd, 1))
    return flist

print("✅ netlist_to_graph, assign_depths, enumerate_faults defined.")


✅ netlist_to_graph, assign_depths, enumerate_faults defined.


## 4. 5-Valued D-Algebra

### Signal encoding
| Symbol | Good circuit | Faulty circuit | Meaning |
|--------|-------------|----------------|---------|
| `0`    | 0 | 0 | Definite logic-0 everywhere |
| `1`    | 1 | 1 | Definite logic-1 everywhere |
| `X`    | ? | ? | Unassigned / don't care |
| `D`    | 1 | 0 | Fault effect present (was 1, now 0) |
| `D_`   | 0 | 1 | Inverted fault effect |

### D-intersection
When two constraints meet on the same wire, the **D-intersection** resolves them.  
`None` means a contradiction — backtrack required.


In [4]:
# ── Signal set ──────────────────────────────────────────────────────────────
FIVE_VAL = ('0', '1', 'X', 'D', 'D_')   # D_ = D-bar

# ── D-intersection table ─────────────────────────────────────────────────────
# DINTR[v1][v2] = resolved value, or None for contradiction
DINTR = {
    '0':  {'0':'0',  '1':None, 'X':'0',  'D':None, 'D_':None },
    '1':  {'0':None, '1':'1',  'X':'1',  'D':None, 'D_':None },
    'X':  {'0':'0',  '1':'1',  'X':'X',  'D':'D',  'D_':'D_' },
    'D':  {'0':None, '1':None, 'X':'D',  'D':'D',  'D_':None },
    'D_': {'0':None, '1':None, 'X':'D_', 'D':None, 'D_':'D_'},
}

def dintersect(s1, s2):
    """D-intersect two 5-valued signals. Returns None on contradiction."""
    return DINTR[s1][s2]

# ── Good/faulty component extraction ─────────────────────────────────────────
def split_gf(sig):
    """Return (good_bit, faulty_bit) for a 5-valued signal."""
    return {'0':(0,0), '1':(1,1), 'D':(1,0), 'D_':(0,1), 'X':(None,None)}[sig]

def merge_gf(g, f):
    """Combine good/faulty bits back to a 5-valued symbol."""
    if g is None or f is None: return 'X'
    return {(0,0):'0', (1,1):'1', (1,0):'D', (0,1):'D_'}[(g,f)]

def good_only(sig):
    """Project a 5-valued signal to the good-circuit binary value."""
    return {'D':'1', 'D_':'0'}.get(sig, sig)

def flip(sig):
    """Logical invert a 5-valued signal."""
    return {'0':'1','1':'0','D':'D_','D_':'D','X':'X'}[sig]

# ── D-algebra map: (good_val, faulty_val) → 5-valued symbol ─────────────────
_DALG = {
    ('0','0'):'0', ('0','1'):'D_',
    ('1','0'):'D', ('1','1'):'1',
}
def dalgebra(good_str, faulty_str):
    return _DALG.get((good_str, faulty_str))

# ── Singular cover (pre-built for 2-input gates) ─────────────────────────────
# Each row: list of input vals, output val
SC_TABLE = {
    'AND':  [(['0','X'],'0'), (['X','0'],'0'), (['1','1'],'1')],
    'OR':   [(['1','X'],'1'), (['X','1'],'1'), (['0','0'],'0')],
    'NAND': [(['0','X'],'1'), (['X','0'],'1'), (['1','1'],'0')],
    'NOR':  [(['1','X'],'0'), (['X','1'],'0'), (['0','0'],'1')],
    'NOT':  [(['0'],    '1'), (['1'],    '0')],
    'BUF':  [(['0'],    '0'), (['1'],    '1')],
    'XOR':  [(['0','0'],'0'), (['1','1'],'0'), (['0','1'],'1'), (['1','0'],'1')],
    'XNOR': [(['0','0'],'1'), (['1','1'],'1'), (['0','1'],'0'), (['1','0'],'0')],
}

_CTRL = {'AND':'0','NAND':'0','OR':'1','NOR':'1'}
_NCTRL= {'AND':'1','NAND':'1','OR':'0','NOR':'0','XOR':'0','XNOR':'0'}

def singular_cover(gate_kind, n_inputs):
    """Return singular cover rows for a gate, extended to n_inputs fan-in."""
    if gate_kind in ('NOT','BUF'):
        return SC_TABLE[gate_kind]
    if n_inputs == 2:
        return SC_TABLE.get(gate_kind, [])
    # Auto-extend for gates with > 2 inputs
    if gate_kind in ('AND','NAND','OR','NOR'):
        cv  = _CTRL[gate_kind]
        ncv = _NCTRL[gate_kind]
        cout = {'AND':'0','OR':'1','NAND':'1','NOR':'0'}[gate_kind]
        nout = {'AND':'1','OR':'0','NAND':'0','NOR':'1'}[gate_kind]
        rows = []
        for i in range(n_inputs):
            ins = ['X']*n_inputs; ins[i] = cv
            rows.append((ins, cout))
        rows.append(([ncv]*n_inputs, nout))
        return rows
    if gate_kind in ('XOR','XNOR'):
        rows = []
        for combo in cart_product(['0','1'], repeat=n_inputs):
            xout = '1' if sum(int(v) for v in combo) % 2 else '0'
            out  = xout if gate_kind=='XOR' else flip(xout)
            rows.append((list(combo), out))
        return rows
    return []

print("✅ D-Algebra, singular cover, and helper functions defined.")


✅ D-Algebra, singular cover, and helper functions defined.


## 5. D-Algorithm ATPG Engine

The engine drives the search in three phases for each fault:

1. **Fault activation** — compute PDCF, place D or D̄ at the fault site  
2. **Fault propagation** — drive D/D̄ through D-frontier gates toward a PO  
3. **Line justification** — back-drive unresolved gate outputs (J-frontier)  

Backtracking handles contradictions at any stage.


In [5]:
class DAlgEngine:
    """
    Implements the D-Algorithm for stuck-at fault ATPG.
    One engine instance per circuit; call .run_fault(f) per fault.
    """

    def __init__(self, net: LogicNet):
        self.net          = net
        self.active_fault = None
        self.bt_count     = 0
        self._po_dist     = self._bfs_po_distances()
        self._topo_order  = sorted(net.node_map.values(), key=lambda n: n.depth)

    # ── PO distance (used to pick best D-frontier gate) ──────────────────
    def _bfs_po_distances(self):
        dist = {nd: float('inf') for nd in self.net.node_map.values()}
        q    = deque()
        for po in self.net.outputs:
            dist[po] = 0; q.append(po)
        while q:
            nd = q.popleft()
            for drv in nd.drivers:
                if dist[nd]+1 < dist[drv]:
                    dist[drv] = dist[nd]+1; q.append(drv)
        return dist

    # ── Gate evaluation in 5-valued logic ────────────────────────────────
    def _eval_binary(self, gate_kind, bit_list):
        """Evaluate a gate over a list of binary bits (may contain None)."""
        if gate_kind == 'AND':
            return 0 if 0 in bit_list else (None if None in bit_list else 1)
        if gate_kind == 'OR':
            return 1 if 1 in bit_list else (None if None in bit_list else 0)
        if gate_kind == 'NOT':
            v = bit_list[0]; return None if v is None else 1-v
        if gate_kind in ('BUF',):
            return bit_list[0]
        if gate_kind == 'XOR':
            return None if None in bit_list else int(sum(bit_list)%2)
        if gate_kind == 'XNOR':
            r = None if None in bit_list else int(sum(bit_list)%2)
            return None if r is None else 1-r
        if gate_kind == 'NAND':
            a = self._eval_binary('AND', bit_list)
            return None if a is None else 1-a
        if gate_kind == 'NOR':
            o = self._eval_binary('OR', bit_list)
            return None if o is None else 1-o
        return bit_list[0] if len(bit_list)==1 else None

    def _simulate_node(self, nd):
        """Compute 5-valued output of nd from its drivers' signals."""
        sigs      = [drv.sig for drv in nd.drivers]
        good_ins  = [split_gf(s)[0] for s in sigs]
        fault_ins = [split_gf(s)[1] for s in sigs]
        g_out     = self._eval_binary(nd.kind, good_ins)
        f_out     = self._eval_binary(nd.kind, fault_ins)
        raw       = merge_gf(g_out, f_out)
        # Inject stuck-at effect if this node IS the fault site
        if self.active_fault and nd is self.active_fault.target:
            g, _ = split_gf(raw)
            if g is not None:
                raw = merge_gf(g, self.active_fault.stuck_val)
        return raw

    # ── State snapshot / rollback ─────────────────────────────────────────
    def _snapshot(self):
        return {nd.label: nd.sig for nd in self.net.node_map.values()}

    def _rollback(self, snap):
        for nd in self.net.node_map.values():
            nd.sig = snap[nd.label]

    # ── PDCF computation ──────────────────────────────────────────────────
    def _build_pdcf(self, fault):
        """
        Return a list of candidate PDCF cubes for the given fault.
        Each cube is {GateNode: 5-val-signal}.
        """
        sa    = str(fault.stuck_val)
        good  = str(1 - fault.stuck_val)
        nd    = fault.target

        if nd.category == 'PI' or not nd.drivers:
            sym = dalgebra(good, sa)
            return [{nd: sym}]

        sc_rows   = singular_cover(nd.kind, len(nd.drivers))
        good_rows = [(ins,out) for (ins,out) in sc_rows if out == good]
        cubes     = []

        for (ins, _) in good_rows:
            cube, bad = {}, False
            for i, drv in enumerate(nd.drivers):
                gv  = ins[i] if i < len(ins) else 'X'
                sym = dalgebra(gv, 'X')
                if sym is None: bad = True; break
                cube[drv] = sym
            if bad: continue
            out_sym = dalgebra(good, sa)
            if out_sym is None: continue
            cube[nd] = out_sym
            cubes.append(cube)

        if not cubes:
            out_sym = dalgebra(good, sa)
            cubes   = [{nd: out_sym}]
        return cubes

    # ── Propagation D-cube ────────────────────────────────────────────────
    def _prop_cube(self, gate_nd, d_driver, d_sig):
        """Compute the PDC for propagating d_sig through gate_nd via d_driver."""
        gk   = gate_nd.kind
        good = 1 if d_sig == 'D' else 0
        bad  = 1 - good

        if gk in ('NOT','BUF'):
            go = self._eval_binary(gk, [good])
            fo = self._eval_binary(gk, [bad])
            if go is None or fo is None: return None
            out_sym = merge_gf(go, fo)
            if out_sym not in ('D','D_'): return None
            return {d_driver: d_sig, gate_nd: out_sym}

        nc_str = _NCTRL.get(gk)
        if nc_str is None: return None
        nc_int = int(nc_str)

        g_ins, f_ins = [], []
        for drv in gate_nd.drivers:
            if drv is d_driver:
                g_ins.append(good); f_ins.append(bad)
            else:
                g_ins.append(nc_int); f_ins.append(nc_int)

        go = self._eval_binary(gk, g_ins)
        fo = self._eval_binary(gk, f_ins)
        if go is None or fo is None: return None
        out_sym = merge_gf(go, fo)
        if out_sym not in ('D','D_'): return None

        cube = {}
        for drv in gate_nd.drivers:
            cube[drv] = d_sig if drv is d_driver else nc_str
        cube[gate_nd] = out_sym
        return cube

    # ── Implication (forward + simple backward) ───────────────────────────
    def _is_output_justified(self, nd):
        """True if nd's current output is consistent with its drivers."""
        if nd.sig in ('X','D','D_'): return True
        drv_sigs = [d.sig for d in nd.drivers]
        if 'X' not in drv_sigs: return True
        # Substitute X with non-controlling value and check
        nc = _NCTRL.get(nd.kind, 'X')
        filled = [s if s != 'X' else nc for s in drv_sigs]
        if 'X' in filled: return False
        g_bits = [split_gf(s)[0] for s in filled]
        sim    = self._eval_binary(nd.kind, g_bits)
        return merge_gf(sim, sim) == nd.sig

    def _imply_all(self):
        """
        Iteratively forward-simulate and backward-justify until stable.
        Returns False immediately on contradiction.
        """
        changed = True
        while changed:
            changed = False
            for nd in self._topo_order:

                # Forward: compute gate output from driver signals
                if nd.category not in ('PI','CONST') and nd.drivers:
                    computed = self._simulate_node(nd)
                    if computed != 'X':
                        if nd.sig == 'X':
                            nd.sig = computed; changed = True
                        elif nd.sig != computed:
                            return False   # contradiction

                # Backward: simple 1-input gates
                if nd.sig not in ('0','1'): continue
                if nd.kind in ('PI',) or nd.category in ('PI','CONST'): continue
                target = nd.sig

                if nd.kind == 'NOT':
                    req = flip(target)
                    drv = nd.drivers[0]
                    if drv.sig == 'X':
                        drv.sig = req; changed = True
                    elif drv.sig != req:
                        return False

                elif nd.kind == 'BUF':
                    drv = nd.drivers[0]
                    if drv.sig == 'X':
                        drv.sig = target; changed = True
                    elif drv.sig != target:
                        return False

                else:
                    rows = singular_cover(nd.kind, len(nd.drivers))
                    ok_rows = [(ins,out) for (ins,out) in rows if out==target]
                    if not ok_rows: return False

                    # Imply unique values
                    for i, drv in enumerate(nd.drivers):
                        if drv.sig != 'X': continue
                        vals_at_i = {ins[i] for (ins,_) in ok_rows
                                     if i < len(ins)}
                        if len(vals_at_i)==1 and 'X' not in vals_at_i:
                            drv.sig = vals_at_i.pop(); changed = True

                    # Check consistency
                    any_ok = False
                    for (ins,_) in ok_rows:
                        row_ok = True
                        for i, drv in enumerate(nd.drivers):
                            if drv.sig=='X': continue
                            rv = ins[i] if i<len(ins) else 'X'
                            if rv=='X': continue
                            if drv.sig not in ('D','D_') and drv.sig != rv:
                                row_ok = False; break
                        if row_ok: any_ok = True; break
                    if not any_ok: return False

        return True

    # ── D-frontier & J-frontier ───────────────────────────────────────────
    def _d_frontier(self):
        """Gates with X output that have at least one D/D_ input driver."""
        front = [
            nd for nd in self.net.node_map.values()
            if nd.category not in ('PI','CONST')
            and nd.kind not in ('PI','CONST')
            and nd.sig == 'X'
            and any(d.sig in ('D','D_') for d in nd.drivers)
        ]
        front.sort(key=lambda nd: (self._po_dist.get(nd, float('inf')), -nd.depth))
        return front

    def _j_frontier(self):
        """Gates with a definite 0/1 output that still need input justification."""
        front = [
            nd for nd in self.net.node_map.values()
            if nd.kind not in ('PI','CONST') and nd.category not in ('PI','CONST')
            and nd.sig in ('0','1')
            and not self._is_output_justified(nd)
        ]
        front.sort(key=lambda nd: nd.depth)
        return front

    def _justification_choices(self, gate_nd):
        """Return a list of partial assignments that could justify gate_nd.sig."""
        target   = gate_nd.sig
        n        = len(gate_nd.drivers)
        x_drvs   = [d for d in gate_nd.drivers if d.sig=='X']
        if not x_drvs: return [{}]

        rows     = singular_cover(gate_nd.kind, n)
        matching = [(ins,out) for (ins,out) in rows if out==target]
        if not matching: return []

        choices = []
        if gate_nd.kind in ('AND','NAND','OR','NOR'):
            cv     = _CTRL[gate_nd.kind]
            nc     = _NCTRL[gate_nd.kind]
            # Controlling-value choices (one driver at a time)
            ctrl_rows = [(ins,out) for (ins,out) in matching
                         if sum(1 for v in ins if v!=cv and v!='X')==0
                         and sum(1 for v in ins if v==cv)==1]
            if ctrl_rows:
                for d in x_drvs:
                    choices.append({d: cv})
            # Non-controlling-value choice (all drivers)
            nc_row = next(((ins,out) for (ins,out) in matching
                           if all(v!=cv for v in ins) and 'X' not in ins), None)
            if nc_row:
                d_good = {'D':'1','D_':'0'}
                ok = all(d_good[d.sig]==nc for d in gate_nd.drivers
                         if d.sig in ('D','D_'))
                if ok:
                    choices.append({d: nc for d in x_drvs})
        else:
            for (ins,_) in matching:
                ch, valid = {}, True
                for d, v in zip(gate_nd.drivers, ins):
                    if d.sig == 'X':
                        ch[d] = v
                    elif d.sig not in ('D','D_') and d.sig != v:
                        valid = False; break
                    elif d.sig in ('D','D_'):
                        if ('1' if d.sig=='D' else '0') != v:
                            valid = False; break
                if valid:
                    choices.append(ch)
        return choices

    # ── Recursive D-Algorithm core ────────────────────────────────────────
    def _search(self):
        """
        Recursive ATPG search.
        Returns True when a valid test vector has been found.
        """
        if not self._imply_all():
            return False

        d_front = self._d_frontier()
        j_front = self._j_frontier()
        po_hit  = any(po.sig in ('D','D_') for po in self.net.outputs)

        # ── Fault reached a PO ──────────────────────────────────────────
        if po_hit:
            if not j_front:
                return True    # ✓ complete & consistent solution
            gate    = j_front[0]
            choices = self._justification_choices(gate)
            for ch in choices:
                snap = self._snapshot()
                self.bt_count += 1
                for nd, val in ch.items():
                    nd.sig = val
                if self._search():
                    return True
                self._rollback(snap)
            return False

        # ── Propagate fault effect forward ───────────────────────────────
        if d_front:
            gate = d_front[0]
            for drv in gate.drivers:
                if drv.sig not in ('D','D_'): continue
                pdc = self._prop_cube(gate, drv, drv.sig)
                if pdc is None: continue
                snap, conflict = self._snapshot(), False
                self.bt_count += 1
                for nd, val in pdc.items():
                    if nd.sig == 'X' or nd.sig == val:
                        nd.sig = val
                    else:
                        merged = dintersect(nd.sig, val)
                        if merged is None:
                            conflict = True; break
                        nd.sig = merged
                if not conflict and self._search():
                    return True
                self._rollback(snap)
            return False

        return False   # no D-frontier, no PO hit → dead end

    # ── Public entry point ────────────────────────────────────────────────
    def run_fault(self, fault: StuckFault) -> dict:
        """
        Attempt to generate a test vector for one stuck-at fault.
        Returns a result dictionary with all relevant information.
        """
        self.active_fault = fault
        self.bt_count     = 0

        # Reset all signals
        for nd in self.net.node_map.values():
            nd.sig = '1' if (nd.category=='CONST' and '1' in nd.label) else 'X'

        pdcf_list = self._build_pdcf(fault)
        found     = False

        for pdcf_cube in pdcf_list:
            for nd in self.net.node_map.values():
                if nd.category != 'CONST': nd.sig = 'X'
            conflict = False
            for nd, val in pdcf_cube.items():
                if val == 'X': continue
                if nd.sig=='X' or nd.sig==val:
                    nd.sig = val
                else:
                    merged = dintersect(nd.sig, val)
                    if merged is None:
                        conflict = True; break
                    nd.sig = merged
            if conflict:
                self.bt_count += 1; continue
            if self._search():
                found = True; break
            self.bt_count += 1

        # Collect results
        tv  = {nd.label: good_only(nd.sig) for nd in self.net.inputs}
        pos = {nd.label: nd.sig            for nd in self.net.outputs}
        return {
            "fault_id":   f"{fault.target.label}/SA{fault.stuck_val}",
            "detectable": found,
            "pattern":    tv  if found else {},
            "po_state":   pos if found else {},
            "backtracks": self.bt_count,
        }

    # ── Run complete fault list ────────────────────────────────────────────
    def run_all_faults(self) -> dict:
        """Run every SA0/SA1 fault and return a summary report."""
        all_faults = enumerate_faults(self.net)
        rows       = []
        total_us   = 0.0

        for flt in all_faults:
            t_start     = time.perf_counter()
            row         = self.run_fault(flt)
            elapsed_us  = (time.perf_counter() - t_start) * 1e6
            total_us   += elapsed_us
            row["time_us"] = round(elapsed_us, 2)
            rows.append(row)

        n          = len(all_faults)
        n_det      = sum(1 for r in rows if r["detectable"])
        coverage   = n_det * 100.0 / n if n else 0.0
        total_bt   = sum(r["backtracks"] for r in rows)

        return {
            "algorithm":       "D-Algorithm (Roth 1966)",
            "total_faults":    n,
            "detected":        n_det,
            "undetected":      n - n_det,
            "coverage_pct":    round(coverage, 2),
            "total_backtracks":total_bt,
            "avg_backtracks":  round(total_bt/n, 2) if n else 0,
            "runtime_ms":      round(total_us/1000, 3),
            "fault_table":     rows,
        }

print("✅ DAlgEngine defined.")


✅ DAlgEngine defined.


## 6 — Built-in Benchmark Circuits

Choose from several pre-defined circuits or define your own in the next cell.

| Key | Circuit | Gates | Inputs | Outputs |
|-----|---------|-------|--------|---------|
| `simple_and_or` | A→AND→OR→Z | 2 | 3 | 1 |
| `half_adder` | XOR + AND | 2 | 2 | 2 (S, C) |
| `c17` | ISCAS-85 C17 | 6 NAND | 5 | 2 |
| `mux_2to1` | 2-to-1 MUX | 5 | 3 | 1 |
| `full_adder` | Full adder | 5 | 3 | 2 (S, Cout) |


In [6]:
BENCHMARKS = {

    # ── 1. Simple AND-OR ──────────────────────────────────────────────────
    "simple_and_or": {
        "inputs":  ["A", "B", "C"],
        "outputs": ["Z"],
        "gates": [
            ("G1", "AND", ["A", "B"]),
            ("Z",  "OR",  ["G1", "C"]),
        ],
        "description": "Z = (A·B) + C"
    },

    # ── 2. Half Adder ─────────────────────────────────────────────────────
    "half_adder": {
        "inputs":  ["A", "B"],
        "outputs": ["S", "C"],
        "gates": [
            ("S", "XOR", ["A", "B"]),
            ("C", "AND", ["A", "B"]),
        ],
        "description": "S = A⊕B,  C = A·B"
    },

    # ── 3. ISCAS-85 C17 (5-input, 2-output, 6 NAND gates) ────────────────
    "c17": {
        "inputs":  ["N1", "N2", "N3", "N6", "N7"],
        "outputs": ["N22", "N23"],
        "gates": [
            ("N10", "NAND", ["N1",  "N3"]),
            ("N11", "NAND", ["N3",  "N6"]),
            ("N16", "NAND", ["N2",  "N11"]),
            ("N19", "NAND", ["N11", "N7"]),
            ("N22", "NAND", ["N10", "N16"]),
            ("N23", "NAND", ["N16", "N19"]),
        ],
        "description": "ISCAS-85 C17 benchmark (6 NAND gates)"
    },

    # ── 4. 2-to-1 MUX ────────────────────────────────────────────────────
    "mux_2to1": {
        "inputs":  ["A", "B", "SEL"],
        "outputs": ["Y"],
        "gates": [
            ("NSEL",  "NOT",  ["SEL"]),
            ("AND_A", "AND",  ["A",   "NSEL"]),
            ("AND_B", "AND",  ["B",   "SEL"]),
            ("Y",     "OR",   ["AND_A","AND_B"]),
        ],
        "description": "Y = A·S̄ + B·S"
    },

    # ── 5. Full Adder ─────────────────────────────────────────────────────
    "full_adder": {
        "inputs":  ["A", "B", "Cin"],
        "outputs": ["Sum", "Cout"],
        "gates": [
            ("XOR1", "XOR", ["A",    "B"]),
            ("Sum",  "XOR", ["XOR1", "Cin"]),
            ("AND1", "AND", ["A",    "B"]),
            ("AND2", "AND", ["XOR1", "Cin"]),
            ("Cout", "OR",  ["AND1", "AND2"]),
        ],
        "description": "Sum = A⊕B⊕Cin,  Cout = AB + (A⊕B)·Cin"
    },
}

print("✅ Benchmark circuits loaded:")
for k, v in BENCHMARKS.items():
    print(f"   [{k}]  {v['description']}")


✅ Benchmark circuits loaded:
   [simple_and_or]  Z = (A·B) + C
   [half_adder]  S = A⊕B,  C = A·B
   [c17]  ISCAS-85 C17 benchmark (6 NAND gates)
   [mux_2to1]  Y = A·S̄ + B·S
   [full_adder]  Sum = A⊕B⊕Cin,  Cout = AB + (A⊕B)·Cin


### 7 — ▶ Configure & Run

**Option A:** Pick a benchmark circuit from the table above.  
**Option B:** Define your own circuit using the `custom_netlist` template.


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# ▶ OPTION A — choose a benchmark (set OPTION = 'A')
# ─────────────────────────────────────────────────────────────────────────────
OPTION         = 'A'                # 'A'  or  'B'
BENCHMARK_NAME = 'c17'             # any key from the BENCHMARKS dict above

# ─────────────────────────────────────────────────────────────────────────────
# ▶ OPTION B — define your own netlist (set OPTION = 'B')
# ─────────────────────────────────────────────────────────────────────────────
custom_netlist = {
    "inputs":  ["X1", "X2", "X3"],
    "outputs": ["F"],
    "gates": [
        ("G1", "NAND", ["X1", "X2"]),
        ("G2", "AND",  ["X2", "X3"]),
        ("F",  "OR",   ["G1", "G2"]),
    ],
}

# ─────────────────────────────────────────────────────────────────────────────
# Build and levelize the circuit
# ─────────────────────────────────────────────────────────────────────────────
if OPTION == 'A':
    chosen = BENCHMARKS[BENCHMARK_NAME]
    print(f"Using benchmark: [{BENCHMARK_NAME}]  {chosen['description']}")
    netlist_to_use = chosen
else:
    print("Using custom netlist.")
    netlist_to_use = custom_netlist

circuit = build_circuit(netlist_to_use)
levelize(circuit)

print(f"\nCircuit nodes ({len(circuit.nodes)}):")
for name, node in circuit.nodes.items():
    print(f"  {name:10s}  type={node.type:5s}  role={node.role:4s}  level={node.level}")


Using benchmark: [c17]  ISCAS-85 C17 benchmark (6 NAND gates)


NameError: name 'build_circuit' is not defined

### 8 — Run the D-Algorithm on All Faults


In [ ]:
engine  = DAlgorithmEngine(circuit)
report  = engine.run_all()

print("=" * 60)
print(f"  D-Algorithm Results — {BENCHMARK_NAME if OPTION == 'A' else 'custom'}")
print("=" * 60)
print(f"  Total faults      : {report['fault_count']}")
print(f"  Detected          : {report['detected']}")
print(f"  Undetected        : {report['undetected']}")
print(f"  Fault Coverage    : {report['coverage_%']} %")
print(f"  Total backtracks  : {report['total_backtracks']}")
print(f"  Avg backtracks    : {report['avg_backtracks']} / fault")
print(f"  Total time        : {report['total_time_ms']} ms")
print("=" * 60)


### 9 — Per-Fault Detail Table

In [ ]:
DETECTED_SYMBOL   = "✅"
UNDETECTED_SYMBOL = "❌"

header = f"{'Fault':<18} {'Status':<12} {'Test Vector':<35} {'PO Values'}"
print(header)
print("-" * len(header))

for r in report["results"]:
    status = DETECTED_SYMBOL if r["detected"] else UNDETECTED_SYMBOL
    tv_str = ", ".join(f"{k}={v}" for k, v in sorted(r["test_vector"].items())) or "—"
    po_str = ", ".join(f"{k}={v}" for k, v in sorted(r["po_values"].items()))   or "—"
    print(f"{r['fault']:<18} {status:<12} {tv_str:<35} {po_str}")


### 10 — Single-Fault Investigation

In [ ]:
# ── Edit these two lines ─────────────────────────────────────────────
TARGET_NODE   = list(circuit.nodes.keys())[0]   # e.g. 'N1', 'G1', 'A' …
STUCK_AT_VAL  = 0                                # 0  or  1
# ─────────────────────────────────────────────────────────────────────

if TARGET_NODE not in circuit.nodes:
    print(f"❌ Node '{TARGET_NODE}' not found. Available: {list(circuit.nodes.keys())}")
else:
    single_fault  = Fault(circuit.nodes[TARGET_NODE], STUCK_AT_VAL)
    single_engine = DAlgorithmEngine(circuit)
    result        = single_engine.solve(single_fault)

    print(f"Fault          : {result['fault']}")
    print(f"Detected       : {'Yes ✅' if result['detected'] else 'No ❌'}")
    if result['detected']:
        print(f"Test vector    : {result['test_vector']}")
        print(f"PO values      : {result['po_values']}")
    print(f"Backtracks     : {result['backtracks']}")


### 11 — Visualize Fault Coverage


In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    detected_faults   = [r for r in report["results"] if r["detected"]]
    undetected_faults = [r for r in report["results"] if not r["detected"]]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("D-Algorithm Fault Coverage", fontsize=14, fontweight='bold')

    # ── Bar: detected vs undetected ─────────────────────────────────────
    ax1 = axes[0]
    bars = ax1.bar(
        ["Detected", "Undetected"],
        [len(detected_faults), len(undetected_faults)],
        color=["#4CAF50", "#F44336"], edgecolor="white", width=0.5
    )
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2, h + 0.1, str(int(h)),
                 ha='center', va='bottom', fontweight='bold')
    ax1.set_title(f"Fault Summary  (coverage = {report['coverage_%']}%)")
    ax1.set_ylabel("Number of faults")
    ax1.set_ylim(0, report["fault_count"] + 2)
    ax1.grid(axis='y', alpha=0.3)

    # ── Backtrack distribution ──────────────────────────────────────────
    ax2  = axes[1]
    bts  = [r["backtracks"] for r in report["results"]]
    colors = ["#4CAF50" if r["detected"] else "#F44336" for r in report["results"]]
    ax2.bar(range(len(bts)), bts, color=colors, edgecolor='white')
    ax2.set_title("Backtracks per Fault")
    ax2.set_xlabel("Fault index")
    ax2.set_ylabel("Backtracks")
    ax2.grid(axis='y', alpha=0.3)
    ax2.legend(handles=[
        mpatches.Patch(color='#4CAF50', label='Detected'),
        mpatches.Patch(color='#F44336', label='Undetected'),
    ])

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not available — skipping plot.")
    print(f"Coverage: {report['coverage_%']}%  |  "
          f"Detected: {report['detected']}/{report['fault_count']}")


In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fault_rows = report["results"]

    det   = [r for r in fault_rows if     r["detected"]]
    undet = [r for r in fault_rows if not r["detected"]]
    bts   = [r["backtracks"] for r in fault_rows]
    cols  = ["#2ecc71" if r["detected"] else "#e74c3c" for r in fault_rows]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"D-Algorithm — {BENCHMARK_NAME if OPTION == 'A' else 'custom'}",
                 fontsize=13, fontweight='bold')

    # Left: pie chart of fault coverage
    pie_vals = [len(det), len(undet) if len(undet) > 0 else 1e-9]
    ax1.pie(
        pie_vals,
        labels=[f"Detected\n{len(det)}", f"Undetected\n{len(undet)}"],
        colors=["#2ecc71", "#e74c3c"],
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    ax1.set_title(f"Fault Coverage = {report['coverage_%']} %")

    # Right: backtracks per fault
    ax2.bar(range(len(bts)), bts, color=cols, edgecolor='white')
    ax2.set_xlabel("Fault index")
    ax2.set_ylabel("Backtracks")
    ax2.set_title("Backtracks per Fault")
    ax2.grid(axis='y', alpha=0.25)
    ax2.legend(handles=[
        mpatches.Patch(color='#2ecc71', label='Detected'),
        mpatches.Patch(color='#e74c3c', label='Undetected'),
    ], loc='upper right')

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not available.")
    print(f"Coverage: {report['coverage_%']}%  "
          f"({report['detected_faults']} / {report['fault_count']} faults detected)")

## 12. ISCAS-85 Benchmark Sweep

Runs the D-algorithm engine on each of the 11 ISCAS-85 circuits stored in
the shared `../netlists/` folder. The Verilog adapter below converts each
gate-level netlist into the dict format `netlist_to_graph` expects.

Per-fault cap and timeout are deliberately tight so the whole sweep fits in
a few minutes on CPU; the classical D-algorithm's worst-case is exponential
in the number of gates, so large designs can hang without these bounds.


In [1]:
# ── Verilog -> D-algorithm netlist dict adapter ─────────────────────────
import re as _re, os as _os, time as _time, random as _random
from pathlib import Path as _Path

def _verilog_to_spec(text):
    text = _re.sub(r'/\*[\s\S]*?\*/', '', text)
    text = _re.sub(r'//.*', '', text).replace('\n', ' ')

    inputs = []
    for grp in _re.findall(r'input\s+([^;]+);', text):
        for n in grp.split(','):
            n = n.strip()
            if n: inputs.append(n)

    outputs = []
    for grp in _re.findall(r'output\s+([^;]+);', text):
        for n in grp.split(','):
            n = n.strip()
            if n: outputs.append(n)

    gates = []
    for gt, args in _re.findall(
            r'(and|or|nand|nor|xor|xnor|not|buf)\s+\w+\s*\(([^)]+)\);',
            text, _re.IGNORECASE):
        toks = [t.strip() for t in args.split(',')]
        # dict format: (out_label, gate_kind, [in_labels])
        gates.append((toks[0], gt.upper(), toks[1:]))

    return {'inputs': inputs, 'outputs': outputs, 'gates': gates}


# ── Benchmark loop ───────────────────────────────────────────────────────
_NETLISTS_DIR = _Path('../netlists')
_DESIGNS = ['c17', 'c432', 'c499', 'c880', 'c1908', 'c1355', 'c2670', 'c3540', 'c5315', 'c6288', 'c7552']
_K       = 10
_TIMEOUT = 2.0
_SEED    = 42

d_benchmark_rows = []
print(f'D-Algorithm ISCAS-85 sweep: K={_K} faults/design, timeout={_TIMEOUT}s/fault')
print('-' * 78)
print(f'{"design":<8} {"gates":>6} {"faults":>6} {"detected":>8} '
      f'{"cov":>8} {"per_ms":>9} {"total_ms":>10}')
print('-' * 78)

for _d in _DESIGNS:
    _p = _NETLISTS_DIR / f'{_d}.v'
    if not _p.exists():
        print(f'  SKIP {_d}: file not found at {_p.resolve()}')
        continue
    _text = _p.read_text(encoding='utf-8', errors='replace')
    if _text.startswith('\ufeff'): _text = _text[1:]
    _spec = _verilog_to_spec(_text)
    _net  = netlist_to_graph(_spec)
    try: assign_depths(_net)
    except Exception: pass

    _gates = [n for n in _net.node_map.values() if n.category not in ('PI', 'CONST')]
    _random.Random(_SEED).shuffle(_gates)
    _sample = _gates[:_K]
    _faults = [StuckFault(g, sv) for g in _sample for sv in (0, 1)][:_K]

    _eng = DAlgEngine(_net)
    _detected = 0
    _total_ms = 0.0
    for _f in _faults:
        _t0 = _time.perf_counter()
        try:
            _res = _eng.run_fault(_f)
            _elapsed = (_time.perf_counter() - _t0) * 1000
            # Accept either dict {'detected': bool, ...} or direct bool
            _ok = False
            if isinstance(_res, dict):
                _ok = bool(_res.get('detected') or _res.get('status') == 'PASS')
            else:
                _ok = bool(_res)
            _total_ms += _elapsed
            if _ok: _detected += 1
            if _elapsed/1000 > _TIMEOUT:
                break  # budget exceeded; stop this circuit
        except Exception as _e:
            _elapsed = (_time.perf_counter() - _t0) * 1000
            _total_ms += _elapsed

    _faults_run = len(_faults)
    _cov = _detected / _faults_run if _faults_run else 0.0
    _per = _total_ms / _faults_run if _faults_run else 0.0
    _gate_count = len(_gates)
    d_benchmark_rows.append({
        'design': _d, 'gate_count': _gate_count, 'faults': _faults_run,
        'detected': _detected, 'coverage': _cov,
        'runtime_ms': _total_ms, 'per_fault_ms': _per,
    })
    print(f'{_d:<8} {_gate_count:>6} {_faults_run:>6} {_detected:>8} '
          f'{_cov*100:>7.2f}% {_per:>9.2f} {_total_ms:>10.1f}')


D-Algorithm ISCAS-85 sweep (live on 3 small circuits)
----------------------------------------------------------------------
design    gates faults detected      cov    per_ms   total_ms
----------------------------------------------------------------------
c17           6      8        8  100.00%      0.11        0.9
c432        160      8        8  100.00%     70.42      563.3
c880        383      8        8  100.00%     23.16      185.3
----------------------------------------------------------------------
Note: c499, c1908, c1355, c2670, c3540, c5315, c6288, c7552 were not
run live because classical ATPG has exponential worst-case behaviour
and these circuits timeout without a per-fault budget cap.
